**Problem 3**: Suppose we have a 1-DOF arm. We want to control both the position and velocity of the arm as it carries out a motion. How to ensure that the motion remains smooth, the torque is consistent, and the position is precise?

PID itself is a **feedback controller**. It reacts to errors in the system. Looking closely into the time series graphs in the previous section, we can see that as the target changes and the motion is carried out, the error jumps massively and the controller overcorrects before damping down again. To control velocity, we'll need something else.

A way to eliminate that is to add a **feedforward** term. We tell the system that we need only this much energy to go somewhere, and the feedforward term carries us $90%$ of the way. Then, the PID only has to correct small, local errors. For our system, let's try to make feedforward hold the position first. Then, the term is simply:
$$
\tau = mg\dfrac{l}{2} \cos \theta
$$
We will also be modelling some disturbances too. The PID controller alone would take more time and have more error locally around these small pushes. With the feedfoward term and some aggressive $K_d$, we can handle them pretty well:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.01 # Time step
time = 20.0 # Total simulation time
t = np.arange(0, time, dt)

# System state
theta = np.zeros_like(t)
omega = np.zeros_like(t)
u = np.zeros_like(t)      # control input
load = np.zeros_like(t) # disturbance torque

theta[0] = np.pi / 2 
omega[0] = 0.0

# Desired position
target = 0

# Maximum control input 
u_max = 40.0  
integral = 0.0

# Dynamics
b = 0.1  # Damping coefficient
m = 2.0  # Mass of the arm
l = 0.5  # Length of the arm
g = 9.81  # Acceleration due to gravity
I = (1/3) * m * l**2  # Moment of inertia of the arm

p = 0.01  # Probability of a torque disturbance (e.g pushing the arm)

# Stochastic shot disturbance parameters
shot_timer = 0.0
shot_duration = 0.4   # seconds
shot_peak = 5.0  # Torque applied during a shot
shot_rate = 0.5          # average shots per second

# Filter initialization
filtered_derivative = 0.0
alpha = 0
previous_error = 0.0

shot_direction = 0

# Simulation
for k in range(len(t)-1):

    error = target - theta[k]
    integral += error * dt  

    # P controller
    Kp = 30  # Proportional gain
    K_i = 1  # Integral gain
    Kd = 10  # Derivative gain


    raw_derivative = -omega[k]
    filtered_derivative = (alpha * filtered_derivative + (1 - alpha) * raw_derivative)

    u[k] = Kp * error + K_i * integral + Kd * filtered_derivative + m * g * (l/2) * np.cos(target)  # Feedforward term

    # Limit the control input
    u[k] = np.clip(u[k], -u_max, u_max)

    # Apply the load
    if shot_timer <= 0 and np.random.rand() < shot_rate * dt:
        shot_timer = shot_duration
        shot_direction = 1 if np.random.rand() < 0.5 else -1

    if shot_timer > 0:
        elapsed = shot_duration - shot_timer
        load[k] = (shot_direction * shot_peak * np.sin(np.pi * elapsed / shot_duration))
    
        shot_timer -= dt
    else:
        load[k] = 0.0

    ambient_disturbance = np.random.normal(0, 0.1)


    # Angular acceleration
    theta_ddot = (u[k] - b * omega[k] - m * g * (l/2) * np.cos(theta[k]) + ambient_disturbance + load[k]) / I

    # Euler integration
    omega[k+1] = omega[k] + theta_ddot * dt
    theta[k+1] = theta[k] + omega[k+1] * dt

u[-1] = u[-2] # Last control input value for plotting

# Plotting

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

ax1.plot(t, theta)
ax1.axhline(target, color='r', linestyle='--')
ax1.set_ylabel("Angular Position")

ax2.step(t, u, where='post')
ax2.set_ylabel("Control Input")
ax2.set_xlabel("Time")

ax3.step(t, load, where='post')
ax3.set_ylabel("Disturbance Torque")
ax3.set_xlabel("Time")

plt.tight_layout()
plt.show()